# Base Model (Pre-Reconciliation)

This notebook builds the base model using spend and impressions only.
No Granger causality or reconciliation is applied yet.

In [2]:
import pandas as pd
import numpy as np
import os

import tensorflow as tf
import tensorflow_probability as tfp

from meridian import constants
from meridian.data import data_frame_input_data_builder
from meridian.model import model
from meridian.model import spec
from meridian.model import prior_distribution
from meridian.analysis import analyzer
from meridian.analysis import visualizer
from meridian.analysis import summarizer
 #from meridian.analysis import reviewer 
 #from meridian.analysis import optimizer

In [3]:
df = pd.read_csv(
    "https://raw.githubusercontent.com/pstat197/BlueAlpha3-Synergy-Analysis/refs/heads/meridian_modeling/data/monthly_mocha.csv"
)

# Remove all-zero columns
df = df.loc[:, (df != 0).any()]

df.head()

,date,subscriptions,meta_spend,meta_impressions,google_spend,google_impressions,snapchat_spend,snapchat_impressions,tiktok_spend,tiktok_impressions,moloco_spend,moloco_impressions,liveintent_spend,liveintent_impressions,beehiiv_spend,beehiiv_impressions,amazon_spend,amazon_impressions
0,8/4/25,15540,91538.06648,16572258,116667.9945,6473132,94750.04035,3420454,0.0,0,6564.524233,367206,37766.44904,371854,18190.10332,181901,0.0,0
1,7/28/25,14525,93840.18612,25300600,180486.9558,9487127,99447.23218,3235285,0.0,0,18111.083980,820589,38543.27888,347850,20063.91811,200639,0.0,0
2,7/21/25,16880,48403.06780,14099214,200817.3250,7909118,84738.57435,4766750,0.0,0,9714.794608,369806,39697.42202,322865,20828.00074,208280,0.0,0
3,7/14/25,20113,49470.96783,13652072,215770.9242,7789279,83204.40500,4022680,0.0,0,16831.841440,554980,40561.34006,570418,24097.34426,240973,0.0,0
4,7/7/25,16492,48948.28744,10121002,209231.9668,6806878,82642.37271,4532105,0.0,0,17624.908800,894891,40012.42040,483619,19967.60420,199676,0.0,0


In [4]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (74, 18)

Columns:
['date', 'subscriptions', 'meta_spend', 'meta_impressions', 'google_spend', 'google_impressions', 'snapchat_spend', 'snapchat_impressions', 'tiktok_spend', 'tiktok_impressions', 'moloco_spend', 'moloco_impressions', 'liveintent_spend', 'liveintent_impressions', 'beehiiv_spend', 'beehiiv_impressions', 'amazon_spend', 'amazon_impressions']


In [6]:
df_model = df.copy()

df_model["time"] = pd.date_range(
    start="2020-01-01",
    periods=len(df_model),
    freq="D"
).astype(str)

df_model["revenue_per_conversion"] = 1.0
df_model["population"] = 1.0

df_model[["time", "revenue_per_conversion", "population"]].head()

,time,revenue_per_conversion,population
0,2020-01-01,1.0,1.0
1,2020-01-02,1.0,1.0
2,2020-01-03,1.0,1.0
3,2020-01-04,1.0,1.0
4,2020-01-05,1.0,1.0


In [7]:
kpi_col = "subscriptions"  # change if needed

spend_cols = [c for c in df_model.columns if c.endswith("_spend")]

media_cols = []
media_spend_cols = []
channels = []

for spend_col in spend_cols:
    base = spend_col.replace("_spend", "")
    impression_col = f"{base}_impressions"

    if impression_col in df_model.columns:
        media_spend_cols.append(spend_col)
        media_cols.append(impression_col)
        channels.append(base)

print("KPI:", kpi_col)
print("Channels:", channels)
print("Spend cols:", media_spend_cols)
print("Impression cols:", media_cols)

KPI: subscriptions
Channels: ['meta', 'google', 'snapchat', 'tiktok', 'moloco', 'liveintent', 'beehiiv', 'amazon']
Spend cols: ['meta_spend', 'google_spend', 'snapchat_spend', 'tiktok_spend', 'moloco_spend', 'liveintent_spend', 'beehiiv_spend', 'amazon_spend']
Impression cols: ['meta_impressions', 'google_impressions', 'snapchat_impressions', 'tiktok_impressions', 'moloco_impressions', 'liveintent_impressions', 'beehiiv_impressions', 'amazon_impressions']


In [8]:
builder = data_frame_input_data_builder.DataFrameInputDataBuilder(
    kpi_type="non_revenue",
    default_kpi_column=kpi_col,
    default_revenue_per_kpi_column="revenue_per_conversion",
)

builder = (
    builder.with_kpi(df_model)
    .with_revenue_per_kpi(df_model)
    .with_population(df_model)
    .with_media(
        df_model,
        media_cols=media_cols,
        media_spend_cols=media_spend_cols,
        media_channels=channels
    )
)

mmm_data = builder.build()

print("MMM input built successfully")

MMM input built successfully


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/meridian/data/input_data_builder.py:715: UserWarning: The `population` argument is ignored in a nationally aggregated model. It will be reset to [1, 1, ..., 1]
  warnings.warn(


In [9]:
prior = prior_distribution.PriorDistribution(
    roi_m=tfp.distributions.LogNormal(
        loc=0.2,
        scale=0.9,
        name=constants.ROI_M
    )
)

model_spec = spec.ModelSpec(
    prior=prior,
    enable_aks=True
)

mmm = model.Meridian(
    input_data=mmm_data,
    model_spec=model_spec
)

print("Model initialized")

2026-04-30 21:11:28.407387: I external/local_xla/xla/service/service.cc:163] XLA service 0x157554640 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2026-04-30 21:11:28.407658: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1777608688.581809 2871025 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/meridian/model/model.py:74: UserWarning: In a nationally aggregated model, the `media_effects_dist` will be reset to `normal`.
  warnings.warn(


Model initialized


In [10]:
mmm.sample_posterior(
    n_chains=1,
    n_adapt=500,
    n_burnin=500,
    n_keep=500
)

print("Sampling complete")

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/meridian/model/prior_distribution.py:1325: UserWarning: Hierarchical distribution parameters must be deterministically zero for national models. tau_g_excl_baseline has been automatically set to Deterministic(0).
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/meridian/model/prior_distribution.py:1325: UserWarning: Hierarchical distribution parameters must be deterministically zero for national models. eta_m has been automatically set to Deterministic(0).
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/meridian/model/prior_distribution.py:1325: UserWarning: Hierarchical distribution parameters must be deterministically zero for national models. eta_rf has been automatically set to Deterministic(0).
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/meridian/model/pr

Sampling complete


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/arviz/data/inference_data.py:157: UserWarning: trace group is not defined in the InferenceData scheme
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/arviz/data/inference_data.py:1647: UserWarning: trace group is not defined in the InferenceData scheme
  warnings.warn(


In [11]:
analyzer_obj = analyzer.Analyzer(mmm)

roi = analyzer_obj.roi().numpy()
mroi = analyzer_obj.marginal_roi().numpy()
inc = analyzer_obj.incremental_outcome().numpy()

roi = np.squeeze(roi)
mroi = np.squeeze(mroi)
inc = np.squeeze(inc)

roi_mean = roi.mean(axis=0)
roi_p5 = np.percentile(roi, 5, axis=0)
roi_p95 = np.percentile(roi, 95, axis=0)

mroi_mean = mroi.mean(axis=0)
inc_mean = inc.mean(axis=0)

results = pd.DataFrame({
    "channel": channels,
    "ROI_mean": roi_mean,
    "ROI_p5": roi_p5,
    "ROI_p95": roi_p95,
    "Marginal_ROI_mean": mroi_mean,
    "Incremental_mean": inc_mean
})

results

/var/folders/39/3cz51q4d0qd3jwknpmw3cxsr0000gn/T/ipykernel_51242/3379385518.py:1: DeprecationWarning: The `meridian` argument is deprecated and will be removed in a future version. Use `model_context` instead.
  analyzer_obj = analyzer.Analyzer(mmm)


,channel,ROI_mean,ROI_p5,ROI_p95,Marginal_ROI_mean,Incremental_mean
0,meta,0.023320,0.011081,0.042403,0.011345,19592.578125
1,google,0.020203,0.008983,0.034942,0.005074,208061.421875
2,snapchat,0.050016,0.020271,0.086807,0.023935,217189.437500
3,tiktok,0.042619,0.016387,0.082646,0.013417,74698.492188
4,moloco,0.091349,0.038402,0.184352,0.024015,76937.539062
5,liveintent,0.147666,0.052814,0.281961,0.049599,117835.218750
6,beehiiv,0.188618,0.069597,0.353328,0.090397,61899.152344
7,amazon,0.209924,0.083337,0.384140,0.110548,3304.260498


In [12]:
model_diagnostics = visualizer.ModelDiagnostics(mmm)
model_diagnostics.plot_rhat_boxplot()

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tensorflow_probability/python/mcmc/diagnostic.py:580: RuntimeWarning: divide by zero encountered in divide
  return (n / (n - 1.)) * biased_var
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/altair/utils/core.py:268: UserWarning: I don't know how to infer vegalite type from 'empty'.  Defaulting to nominal.
  warnings.warn(


alt.LayerChart(...)

In [13]:
model_fit = visualizer.ModelFit(mmm)
model_fit.plot_model_fit()

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/meridian/analysis/analyzer.py:695: UserWarning: The `aggregate_geos` argument is ignored in the national model. It will be reset to `True`.
  warnings.warn(


alt.LayerChart(...)

In [14]:
best_lags[["target", "driver", "lag", "p_value", "fdr_q"]]

NameError: name 'best_lags' is not defined